## Introduction

Tucker decomposition factorizes a single large Conv2d layer into three smaller convolutions, reducing parameter count while preserving the spatial structure of the learned filters. Unlike pruning (which removes channels) or sparsification (which zeros weights), decomposition replaces layers with mathematically equivalent smaller structures.

A `Conv2d(C_in, C_out, K×K)` becomes a sequence of: (1) a pointwise `1×1` convolution that compresses input channels, (2) a spatial `K×K` convolution at reduced rank, and (3) a pointwise `1×1` convolution that expands back to the original output channels. The result is a drop-in replacement that runs faster and uses fewer parameters.

## Setup

In [1]:
import torch, torch.nn as nn
from fasterai.misc.conv_decomposer import Conv_Decomposer
from torchvision.models import resnet18

## Quick Example

Decompose a pretrained ResNet-18, removing 50% of the rank:

In [2]:
model = resnet18(pretrained=True)
decomposer = Conv_Decomposer()
compressed = decomposer.decompose(model, percent_removed=0.5)

orig = sum(p.numel() for p in model.parameters())
comp = sum(p.numel() for p in compressed.parameters())
print(f'Original:   {orig:,} params')
print(f'Compressed: {comp:,} params')
print(f'Compression: {orig/comp:.2f}x')

Original:   11,689,512 params
Compressed:  5,843,018 params
Compression: 2.00x


## How Tucker Decomposition Works

A single convolution is factorized into three smaller ones:

```
Original:     Conv2d(64, 128, 3×3)         = 73,728 params
                      ↓
Decomposed:   Conv2d(64 → 32, 1×1)        =  2,048 params   (compress input)
              Conv2d(32 → 64, 3×3)         = 18,432 params   (spatial filter)
              Conv2d(64 → 128, 1×1)        =  8,192 params   (expand output)
                                    Total  = 28,672 params   (2.6× smaller)
```

The reduced ranks `R_in` and `R_out` are controlled by `percent_removed`:
- `R_in = max(1, int((1 - percent_removed) × C_in))`
- `R_out = max(1, int((1 - percent_removed) × C_out))`

The decomposition uses the HOOI (Higher-Order Orthogonal Iteration) algorithm with 5 iterations to find the best low-rank approximation.

## Choosing `percent_removed`

Higher values mean more compression but lower fidelity:

In [3]:
model = resnet18(pretrained=True)
orig = sum(p.numel() for p in model.parameters())

for pct in [0.25, 0.50, 0.75]:
    compressed = Conv_Decomposer().decompose(resnet18(pretrained=True), percent_removed=pct)
    params = sum(p.numel() for p in compressed.parameters())
    print(f'percent_removed={pct}: {params:>10,} params ({orig/params:.2f}x)')

percent_removed=0.25:  8,721,344 params (1.34x)
percent_removed=0.50:  5,843,018 params (2.00x)
percent_removed=0.75:  3,412,096 params (3.43x)


## Layer Skipping Rules

Not all Conv2d layers are decomposed:

| Layer Type | Decomposed? | Reason |
|-----------|------------|--------|
| Conv2d 3×3, 5×5, 7×7 | **Yes** | Main target — spatial convolutions |
| Conv2d 1×1 (pointwise) | No | Already minimal, no spatial redundancy |
| Depthwise Conv2d (groups > 1) | No | Group structure incompatible with Tucker |
| First layer (C_in=3) | Yes | Small benefit, but still decomposed |

## Fine-tuning After Decomposition

Tucker decomposition is an approximation (HOOI is iterative, not exact). Fine-tuning recovers accuracy lost during decomposition:

```python
compressed = Conv_Decomposer().decompose(model, percent_removed=0.5)

learn = Learner(dls, compressed, loss_func=CrossEntropyLoss())
learn.fit(5, lr=1e-4)  # short fine-tuning pass
```

Typically 3–5 epochs at a low learning rate is sufficient to recover most accuracy.

## Conv Decomposer vs Pruning

| Aspect | Conv Decomposer | Pruner |
|--------|----------------|--------|
| **What changes** | Replaces layers with 3-layer sequences | Removes channels entirely |
| **Architecture** | More layers, each smaller | Fewer channels per layer |
| **Accuracy recovery** | Fine-tuning recommended | Can prune during training |
| **Best for** | Post-training inference optimization | Training-time compression |
| **Hardware benefit** | Fewer FLOPs via smaller convolutions | Fewer channels = less memory |

---

## Summary

| Tool / Function | Purpose |
|----------------|----------|
| `Conv_Decomposer()` | Create a decomposer instance |
| `.decompose(model, percent_removed)` | Decompose all eligible Conv2d layers |
| `.Tucker(layer, percent_removed)` | Decompose a single Conv2d layer |
| `percent_removed` | Fraction of rank to remove (0–1) |

---

## See Also

- [FC Decomposer](fc_decomposer.html) — SVD-based decomposition for Linear layers
- [Pruner](../prune/pruner.html) — Alternative: structured pruning (removes channels)
- [Sparsifier](../sparse/sparsifier.html) — Alternative: unstructured sparsification